# 🎨 Pyxel Canvas
### 100 Stunning 2D & 3D Patterns in a Single Jupyter Notebook

---

Welcome to **Pyxel Canvas** — a fully interactive visual engine and gallery system.
Select a category, pick a pattern, adjust the controls, and click **RENDER**.

> **Tip:** Run all cells from top to bottom on first launch. Section 0 will verify all dependencies.

---
## Section 0 — Environment Setup
Verify all required libraries are available.

In [ ]:
# ── Section 0: Environment Setup ─────────────────────────────────
import importlib
from pathlib import Path

# Root path — all other paths are relative to this
NOTEBOOK_ROOT = Path.cwd()
print(f"📁 NOTEBOOK_ROOT = {NOTEBOOK_ROOT}")

# Required packages: (pip_name, import_name)
_REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("Pillow", "PIL"),
    ("ipywidgets", "ipywidgets"),
    ("scipy", "scipy"),
    ("tqdm", "tqdm"),
]

print("\n🔧 Checking dependencies...\n")
_missing = []
for pip_name, import_name in _REQUIRED:
    try:
        importlib.import_module(import_name)
        print(f"  ✅ {pip_name} — available")
    except ImportError:
        print(f"  ❌ {pip_name} — MISSING")
        _missing.append(pip_name)

if _missing:
    print(f"\n⚠️  Missing packages: {', '.join(_missing)}")
    print("   Install via MSYS2:  pacman -S mingw-w64-ucrt-x86_64-python-<name>")
    print("   Or via pip:         python -m pip install <name>")
else:
    # Create output directories
    for d in ["exports", "assets", "shaders"]:
        (NOTEBOOK_ROOT / d).mkdir(exist_ok=True)
    print("\n✨ Environment ready!")

---
## Section 1 — Core Engine
Load pattern registry, engines, and utilities into the notebook namespace.

In [ ]:
# ── Section 1: Core Engine ───────────────────────────────────────
import sys
from pathlib import Path

# Make sure our package is importable
_engine_root = str(Path.cwd().parent) if Path.cwd().name == 'visual_engine' else str(Path.cwd())
if _engine_root not in sys.path:
    sys.path.insert(0, _engine_root)

_ve_root = str(Path.cwd()) if Path.cwd().name == 'visual_engine' else str(Path.cwd() / 'visual_engine')
if _ve_root not in sys.path:
    sys.path.insert(0, _ve_root)

# Import the pattern registry
from patterns import PATTERNS, CATEGORIES
from engines import PALETTES
from engines.color_utils import ColorUtils
from utils.export import export_png

print(f"🎨 Loaded {len(PATTERNS)} patterns across {len(CATEGORIES)} categories")
for cat, pats in CATEGORIES.items():
    print(f"   • {cat}: {len(pats)} patterns")

---
## Section 2 — UI Controls
Interactive gallery interface — select a category and pattern, adjust controls, render.

In [ ]:
# ── Section 2: UI Controls ───────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ── Caches ────────────────────────────────────────────────────────
_render_cache   = {}   # renderer instances, keyed by pattern name
_controls_cache = {}   # widget lists, keyed by pattern name — same instances reused everywhere

def _get_renderer(name):
    if name not in _render_cache:
        _render_cache[name] = PATTERNS[name]
    return _render_cache[name]

def _get_controls(name):
    """Return the cached control widgets for a pattern (created once, reused forever)."""
    if name not in _controls_cache:
        _controls_cache[name] = _get_renderer(name).get_controls()
    return _controls_cache[name]

# ── Widgets ────────────────────────────────────────────────────────

category_dd = widgets.Dropdown(
    options=list(CATEGORIES.keys()),
    value=list(CATEGORIES.keys())[0],
    description='Category:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

pattern_dd = widgets.Dropdown(
    options=CATEGORIES[category_dd.value],
    description='Pattern:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

palette_dd = widgets.Dropdown(
    options=list(PALETTES.keys()),
    value='Inferno',
    description='Palette:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

speed_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Speed:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
    readout_format='.1f',
)

resolution_dd = widgets.Dropdown(
    options=['Low', 'Medium', 'High'],
    value='Low',
    description='Resolution:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

render_btn = widgets.Button(
    description='🎨 RENDER',
    button_style='success',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

export_btn = widgets.Button(
    description='💾 EXPORT',
    button_style='info',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

# VBox instead of Output — children replacement is atomic, no clear_output race conditions
pattern_controls_area = widgets.VBox(
    layout=widgets.Layout(
        border='1px solid #333', min_height='40px', padding='8px', margin='4px 0',
    )
)

render_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #444', min_height='200px', padding='8px',
))

status_label = widgets.HTML(
    value='<i style="color:#888;">Select a pattern and click RENDER</i>',
    layout=widgets.Layout(margin='4px 0'),
)

# ── Callbacks ──────────────────────────────────────────────────────

def _on_category_change(change):
    """Switch category — update pattern_dd without double-firing _on_pattern_change."""
    pattern_dd.unobserve(_on_pattern_change, names='value')
    pattern_dd.options = CATEGORIES[change['new']]
    pattern_dd.value   = CATEGORIES[change['new']][0]
    pattern_dd.observe(_on_pattern_change, names='value')
    _on_pattern_change({'new': pattern_dd.value})   # fire exactly once

def _on_pattern_change(change):
    """Swap the controls panel — atomic VBox.children replace, no flicker or doubling."""
    controls = _get_controls(change['new'])
    pattern_controls_area.children = (
        controls if controls
        else [widgets.Label('No pattern-specific controls.')]
    )

_last_fig = [None]

def _on_render(btn):
    """Render selected pattern — plt.show() inside render() is the only display event."""
    with render_output:
        clear_output(wait=True)
        name = pattern_dd.value
        status_label.value = f'<b style="color:#4fc3f7;">⏳ Rendering: {name}...</b>'
        renderer = _get_renderer(name)

        # Read values from the DISPLAYED controls (same cached instances the user adjusted).
        extra_kwargs = {}
        for ctrl in _get_controls(name):
            if hasattr(ctrl, 'description') and hasattr(ctrl, 'value'):
                key = ctrl.description.strip(':').strip().lower().replace(' ', '_')
                extra_kwargs[key] = ctrl.value

        try:
            # render() calls plt.show() then plt.close(fig) — that is the one and only
            # display event.  No sink, no manual display() call needed here.
            renderer.render(
                resolution=resolution_dd.value,
                palette=palette_dd.value,
                speed=speed_slider.value,
                **extra_kwargs,
            )
            # renderer._fig is assigned by _create_figure() and the Python object
            # remains valid for savefig() even after plt.close() deregisters it.
            _last_fig[0] = renderer._fig
            status_label.value = f'<b style="color:#81c784;">✅ Rendered: {name}</b>'
        except Exception as e:
            status_label.value = f'<b style="color:#e57373;">❌ Error: {e}</b>'
            import traceback
            traceback.print_exc()

def _on_export(btn):
    """Export the last rendered figure to PNG."""
    if _last_fig[0] is not None:
        path = export_png(_last_fig[0], pattern_dd.value)
        status_label.value = f'<b style="color:#81c784;">💾 Exported → {path}</b>'
    else:
        status_label.value = '<b style="color:#ffb74d;">⚠️ Nothing to export — render a pattern first.</b>'

# Wire up callbacks
category_dd.observe(_on_category_change, names='value')
pattern_dd.observe(_on_pattern_change,   names='value')
render_btn.on_click(_on_render)
export_btn.on_click(_on_export)

# ── Layout ─────────────────────────────────────────────────────────

header = widgets.HTML(value='''
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
                padding:16px 24px; border-radius:12px; margin-bottom:12px;">
        <h2 style="color:#e0e0e0; margin:0;">🎨 Pyxel Canvas</h2>
        <p style="color:#888; margin:4px 0 0;">Select a category → pick a pattern → click RENDER</p>
    </div>
''')

selectors_row = widgets.HBox([category_dd, pattern_dd],
                              layout=widgets.Layout(gap='12px'))
controls_row  = widgets.HBox([palette_dd, speed_slider, resolution_dd],
                              layout=widgets.Layout(gap='12px'))
buttons_row   = widgets.HBox([render_btn, export_btn],
                              layout=widgets.Layout(gap='12px'))

ui = widgets.VBox([
    header,
    selectors_row,
    controls_row,
    pattern_controls_area,
    buttons_row,
    status_label,
    render_output,
], layout=widgets.Layout(padding='8px'))

display(ui)
_on_pattern_change({'new': pattern_dd.value})

---
## Section 3 — Render Section
The RENDER button above triggers rendering. No separate cell needed — the UI handles it.

---

## Section 4 — Pattern Implementations

All 100 patterns are implemented as classes in the `patterns/` module files.
Concept Notes for each completed pattern are listed below.

### How This Works — Mandelbrot Fractal Explorer

**Core Idea**  
The Mandelbrot set is the set of complex numbers $c$ for which the sequence produced by iterating $z_{n+1} = z_n^2 + c$ (starting from $z_0 = 0$) remains bounded — that is, never escapes to infinity. Points inside the set are colored black; points outside are colored by how quickly they escape, producing the iconic fractal boundary.

**Mathematics**  
The iteration is:
$$z_{n+1} = z_n^2 + c, \quad z_0 = 0, \quad c \in \mathbb{C}$$
Each pixel maps to a value of $c = x + iy$, where $x$ and $y$ are the real and imaginary axes. The escape condition is $|z_n| > 2$. The `max_iter` control sets the iteration ceiling: higher values reveal finer boundary detail at the cost of computation time. Smooth coloring uses the formula $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$ to eliminate banding.

**Logic in the Code**  
1. A 2D numpy array of complex numbers is constructed via `meshgrid` over the viewport defined by `center` and `zoom`.  
2. The iteration $z = z^2 + c$ is applied element-wise using numpy broadcasting — no Python loop over pixels.  
3. An escape mask tracks which pixels have crossed $|z| > 2$. Those pixels are frozen and excluded from subsequent steps.  
4. The smooth escape-count array is passed through a `ColorUtils` colormap to produce the final image.

**Interesting Properties**  
The Mandelbrot set is infinitely self-similar: zooming into any part of the boundary reveals structures that resemble — but are not identical to — the whole set. The cardioid shape of the main body is the locus of $c$ values where $z$ converges to a fixed point, while the circular bulb to its left corresponds to period-2 orbits.

### How This Works — Julia Set Animator

**Core Idea**  
A Julia set is the companion fractal to the Mandelbrot set. Instead of varying $c$ per pixel and starting from $z_0 = 0$, a Julia set fixes $c$ and varies the starting point $z_0$ across the complex plane. Each value of $c$ produces a different Julia set — connected (filled) when $c$ is inside the Mandelbrot set, and a Cantor dust when $c$ is outside.

**Mathematics**  
$$z_{n+1} = z_n^2 + c, \quad z_0 = x + iy \text{ (varies per pixel)}, \quad c \text{ is fixed}$$
The escape condition and smooth coloring are identical to Mandelbrot: $|z_n| > 2$ and $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$. The `c_real` and `c_imag` sliders control the fixed constant, allowing exploration of radically different fractal shapes.

**Logic in the Code**  
1. A complex grid of $z_0$ values is built from the viewport extent.  
2. The same vectorized escape-time loop runs, but with a constant $c$ added at each step.  
3. The resulting escape array is rendered through the selected colormap.

**Interesting Properties**  
Moving the $c$ parameter along the boundary of the Mandelbrot set produces Julia sets that transition from connected to disconnected — the so-called Douady rabbit ($c \approx -0.123 + 0.745i$) and the dendrite ($c = i$) are famous examples. The Mandelbrot set is, in fact, a map of all possible Julia set topologies.

### How This Works — Sierpinski Triangle

**Core Idea**  
The Sierpinski triangle is a self-similar fractal formed by recursively removing the central inverted triangle from an equilateral triangle. After infinite iterations the remaining area is zero, but the structure retains infinite detail. It appears naturally in Pascal's triangle (odd entries) and cellular automata (Rule 90).

**Mathematics**  
At each recursion level, every triangle is replaced by three sub-triangles at half scale, positioned at the three corners. The fractal dimension is:
$$D = \frac{\log 3}{\log 2} \approx 1.585$$
At depth $d$, the number of filled triangles is $3^d$ and each has side length $2^{-d}$ of the original. The `depth` control directly sets $d$.

**Logic in the Code**  
1. Start with the three vertices of an equilateral triangle.  
2. Recursively subdivide: compute midpoints of each edge, forming three corner triangles (discarding the center).  
3. At the base case ($d = 0$), store the triangle as a `[v0, v1, v2]` list.  
4. All triangles are drawn as filled `Polygon` patches with colors sampled from a gradient.

**Interesting Properties**  
The Sierpinski triangle can also be generated by the chaos game: repeatedly pick a random vertex and move halfway toward it from the current point. Despite the randomness, the attractor is deterministically the Sierpinski triangle.

### How This Works — Koch Snowflake

**Core Idea**  
The Koch snowflake is constructed by repeatedly replacing the middle third of every line segment with an equilateral triangular bump. Starting from an equilateral triangle, this process produces a closed curve of infinite length enclosing a finite area — a key early example of a fractal curve.

**Mathematics**  
Each iteration multiplies the number of segments by 4 and divides each segment's length by 3. After $n$ iterations:
$$L_n = L_0 \cdot \left(\frac{4}{3}\right)^n \to \infty, \qquad A_n = A_0 \cdot \frac{8}{5}\left(1 - \frac{3}{9^n}\right)$$
The fractal dimension of the Koch curve is $D = \log 4 / \log 3 \approx 1.262$. The `iterations` slider controls $n$.

**Logic in the Code**  
1. Define three vertices of an equilateral triangle as the snowflake base.  
2. For each edge, recursively subdivide: split into thirds, compute the peak point by rotating the middle third by $60°$ using a 2D rotation.  
3. Collect all points and draw each line segment with a color from the gradient, producing a rainbow Koch snowflake.

**Interesting Properties**  
The Koch snowflake is nowhere differentiable — it has no tangent at any point. It was one of the first curves shown to be continuous everywhere but differentiable nowhere, predating Mandelbrot's formalization of fractal geometry by decades.

### How This Works — Penrose Tiling

**Core Idea**  
A Penrose tiling is an aperiodic tiling — it fills the plane without gaps or overlaps, yet never repeats. Discovered by Roger Penrose in 1974, the P3 variant uses two rhombus shapes (thin and thick) derived from Robinson triangles. The pattern exhibits five-fold rotational symmetry, which is forbidden in periodic crystals.

**Mathematics**  
The construction uses the golden ratio $\varphi = (1 + \sqrt{5})/2 \approx 1.618$. Robinson triangle deflation splits each triangle into smaller ones using subdivision points at $1/\varphi$ along edges:
$$P = A + \frac{B - A}{\varphi}, \qquad Q = B + \frac{A - B}{\varphi}, \qquad R = B + \frac{C - B}{\varphi}$$
Thin (acute) triangles split into 2 sub-triangles; thick (obtuse) triangles split into 3. The `generations` control sets the number of deflation rounds.

**Logic in the Code**  
1. Start with a sun configuration: 10 isosceles triangles arranged in a decagon (alternating orientations).  
2. For each generation, apply the deflation rules — thin → 2 children, thick → 3 children.  
3. Draw all resulting triangles as filled polygons, using two palette colors to distinguish thin from thick rhombi.

**Interesting Properties**  
Penrose tilings are quasicrystals in 2D: they have long-range order (sharp diffraction peaks with five-fold symmetry) but no translational periodicity. The ratio of thick to thin rhombi converges to $\varphi$ as the tiling grows — the golden ratio appears at every scale.

### How This Works — Voronoi Diagram

**Core Idea**
A Voronoi diagram partitions a plane into regions based on proximity to a set of seed points. Each region — called a Voronoi cell — contains every point closer to its seed than to any other. The result is a natural tessellation found everywhere in nature: giraffe skin, soap bubble clusters, and cellular tissue.

**Mathematics**
For seeds $P = \{p_1, \ldots, p_n\}$, the Voronoi cell of $p_i$ is:
$$V(p_i) = \{ x \in \mathbb{R}^2 \mid \|x - p_i\| \leq \|x - p_j\| \;\forall\, j \neq i \}$$
The boundary between two adjacent cells is the perpendicular bisector of the segment joining their seeds. The *Points* slider controls $n$; more seeds produce smaller, more uniform cells.

**Logic in the Code**
1. $n$ random seeds are placed in $[0,1]^2$ using a seeded NumPy RNG for reproducibility.
2. A `scipy.spatial.cKDTree` spatial index is built on the seeds.
3. The canvas is rasterized as a flat grid of $\text{res}^2$ pixels; one `tree.query` call assigns the nearest seed to every pixel simultaneously — no Python loop over pixels.
4. The integer assignment array indexes into the palette gradient to produce the final RGB image rendered via `imshow`.

**Interesting Properties**
Voronoi diagrams are the geometric dual of Delaunay triangulations: connecting adjacent seeds whose cells share a border produces the Delaunay graph, which maximises the minimum triangle angle across the plane. Lloyd's algorithm — iteratively replacing each seed with its cell centroid — converges toward perfectly uniform hexagonal packing.

### How This Works — Fibonacci Spiral

**Core Idea**
The Fibonacci spiral places $n$ points by rotating each successive point by the golden angle — the strategy used by sunflower seeds, pine cones, and pineapple scales to pack the maximum number of seeds per unit area without gaps or clumping.

**Mathematics**
Each seed $i$ is placed at polar coordinates:
$$r_i = \sqrt{i/n}, \qquad \theta_i = i \cdot \phi_g$$
where the **golden angle** is:
$$\phi_g = \pi(3 - \sqrt{5}) \approx 137.508°$$
The irrational nature of $\phi_g$ — derived from the golden ratio $\varphi = \tfrac{1+\sqrt{5}}{2}$ — ensures no two seeds ever land on the same radial spoke, so every new seed fills the largest visible gap.

**Logic in the Code**
1. Indices $i = 0, \ldots, n{-}1$ are generated as a NumPy array.
2. Radii are $\sqrt{i/n}$ (uniform area distribution) and angles are $i \cdot \phi_g$.
3. Cartesian coordinates follow from standard polar conversion; the palette gradient maps $i$ linearly to colour so inner and outer seeds receive different hues.
4. `ax.scatter` renders all $n$ points in a single vectorised call.

**Interesting Properties**
Plotting consecutive Fibonacci-number multiples of seeds reveals the clockwise and counter-clockwise spiral arms visible in real sunflowers. Displacing the angle by even 0.1° away from $\phi_g$ causes the pattern to collapse into coarse radial spokes — demonstrating why the golden angle is the uniquely optimal packing angle.

### How This Works — Dragon Curve

**Core Idea**
The Dragon Curve is a fractal obtained by repeatedly folding a strip of paper in half in the same direction, then unfolding it so every crease stands at 90°. After $n$ folds the unfolded boundary traces a self-avoiding curve that tiles the plane when four copies are assembled around a common point.

**Mathematics**
The turn sequence $T_n$ — recording right (R) or left (L) turns at each step — obeys the recurrence:
$$T_{n+1} = T_n \;\|\; [\text{R}] \;\|\; \overline{T_n^{\;\text{rev}}}$$
where $\overline{\cdot}$ flips all turns (R$\leftrightarrow$L) and $\cdot^{\text{rev}}$ reverses the list. Starting from $T_1 = [\text{R}]$, iteration $n$ yields $2^n - 1$ turns and $2^n$ unit segments. The Hausdorff dimension of the Dragon Curve boundary is $\tfrac{\log 4}{\log 3} \approx 1.52$.

**Logic in the Code**
1. The turn list is built iteratively: each pass appends the current list, a right turn, then the reversed-and-negated list.
2. A direction index (0–3, cycling right/up/left/down) is updated for each turn and used to step the path one unit forward.
3. The path array is converted into a `(N{-}1, 2, 2)` segment array and passed to `LineCollection` — all $2^n{-}1$ segments drawn in a single renderer call, essential for $n \geq 12$.

**Interesting Properties**
Despite its jagged appearance the Dragon Curve never self-intersects, yet in the limit $n \to \infty$ it fills a bounded fractal region completely. Four copies rotated by multiples of 90° tile the plane perfectly — a property exploited by Jurassic Park's book cover art and numerous fractal installations.

### How This Works — Hilbert Curve

**Core Idea**
The Hilbert Curve is a continuous space-filling curve that visits every cell of an $n \times n$ grid exactly once while moving only to directly adjacent cells. Invented by David Hilbert in 1891, it preserves spatial locality so well that nearby points in 2D remain nearby along the 1D index — making it invaluable for database locality, image compression, and CPU cache optimisation.

**Mathematics**
At order $k$ the curve covers a $2^k \times 2^k$ grid visiting all $4^k$ cells. Each order is built from four rotated/reflected copies of the previous order. The index-to-coordinate mapping decodes $d \in [0,4^k)$ by reading successive 2-bit pairs $(r_x, r_y)$:
$$r_x = \bigl\lfloor d/2 \bigr\rfloor \bmod 2, \quad r_y = (r_x \oplus d) \bmod 2$$
with a conditional 90° rotation applied at each level before accumulating the grid offset.

**Logic in the Code**
1. `_hilbert_xy` loops over all $4^k$ indices; for each it iterates over $k$ bit-pair levels, applying the rotation and offset to accumulate $(x, y)$.
2. The resulting $(4^k, 2)$ coordinate array is stacked into $(4^k{-}1, 2, 2)$ segments for a `LineCollection` draw — no individual `ax.plot` calls.
3. The palette gradient maps position along the 1D index to colour, so the progression of the space-filling traversal is immediately visible.

**Interesting Properties**
The Hilbert Curve's locality preservation is quantified: the Euclidean distance between two grid cells never exceeds $O(\sqrt{\Delta d})$ where $\Delta d$ is their 1D index distance — far superior to the $O(n)$ worst case of a row-scan. Higher orders reveal the same rotated U-shape repeating at every scale, a canonical example of exact geometric self-similarity.

### How This Works — L-System Tree

**Core Idea**
An L-System (Lindenmayer System) is a formal string-rewriting grammar that applies production rules to every symbol in parallel at each step. When the resulting string is interpreted as turtle-graphics commands, it produces branching structures that closely resemble real plants. Aristid Lindenmayer introduced them in 1968 to model algae cell growth.

**Mathematics**
The system uses axiom $\omega = X$ with two simultaneous rules:
$$X \;\to\; F{+}[[X]{-}X]{-}F[{-}FX]{+}X \qquad F \;\to\; FF$$
After $n$ iterations the number of $F$ symbols (drawable segments) grows as $\approx 896$ at $n=7$ and the string length as $O(2^n)$. The turtle interprets: $F$ = step forward; $+/-$ = turn $\pm\delta$ (the *Angle* control); $[$ = push state; $]$ = pop state. The branching angle $\delta$ is the dominant shape parameter: small $\delta$ yields narrow spruce forms; large $\delta$ yields spreading oaks.

**Logic in the Code**
1. The axiom is expanded $n$ times using a single `str.join` pass per iteration — substituting every character via a dictionary lookup.
2. A turtle state $(x, y, \theta, \text{depth})$ is maintained with a stack. Each `F` draws a segment and records the current stack depth.
3. Segments are coloured by depth via the palette so trunk segments (depth 0) and fine twigs (deepest depth) receive distinct hues, giving a natural light-to-shadow gradient.
4. All segments are rendered in one `LineCollection` call.

**Interesting Properties**
This rule set is Prusinkiewicz and Lindenmayer's "branching tree" from *The Algorithmic Beauty of Plants* (1990). Changing only the angle from 25° to 30° dramatically shifts the silhouette from a symmetric fern to a wind-swept deciduous shape, illustrating how a single parameter controls emergent biological form.

### How This Works — Apollonius Gasket

**Core Idea**
The Apollonius Gasket is a fractal constructed by repeatedly packing circles into the curved triangular gaps formed by three mutually tangent circles. Starting from three equal circles inside a bounding circle, each gap is filled with a new tangent circle, then the three new gaps around that circle are filled, and so on infinitely. The result is a fractal arrangement of tangent circles whose boundary has Hausdorff dimension ≈ 1.305.

**Mathematics**
The key tool is **Descartes' Circle Theorem** (1643): if four circles are mutually tangent with curvatures $k_1, k_2, k_3, k_4$ (where $k = 1/r$ and $k < 0$ for the outer bounding circle), then:
$$(k_1+k_2+k_3+k_4)^2 = 2(k_1^2+k_2^2+k_3^2+k_4^2)$$
Solving for the unknown curvature gives:
$$k_4 = k_1+k_2+k_3 \pm 2\sqrt{k_1 k_2 + k_2 k_3 + k_1 k_3}$$
The **Extended Descartes theorem** adds center positions (as complex numbers $z = x+iy$):
$$k_4 z_4 = k_1 z_1 + k_2 z_2 + k_3 z_3 \pm 2\sqrt{k_1 k_2 z_1 z_2 + k_2 k_3 z_2 z_3 + k_1 k_3 z_1 z_3}$$
The $\pm$ sign in both formulas is the **same** — the two solutions correspond to the two circles tangent to a given triple. The three equal inner circles have curvature $k_s = 1 + 2/\sqrt{3}$ (from Descartes applied to the initial quadruple $k_0{=}{-}1, k_1{=}k_2{=}k_3{=}k_s$).

**Logic in the Code**
1. The gasket is built via BFS. The initial four circles seed a queue with 4 "gap triples": $(c_0,c_1,c_2)$, $(c_0,c_1,c_3)$, $(c_0,c_2,c_3)$, $(c_1,c_2,c_3)$.
2. For each triple, both Descartes solutions are computed. Circles with negative/zero curvature or outside the bounding disk are rejected.
3. Each new circle $c_4$ spawns 3 new triples — the 3 new gaps it creates with the parent triple. A rounding-based key prevents duplicates.
4. BFS stops when the exploration depth or a circle-count cap is reached. Circles are drawn largest-first and colored by depth level.

**Interesting Properties**
The Apollonius Gasket is a classic example of a **fractal limit set**. The total area of all circles converges to the area of the bounding circle, but the boundary between filled and unfilled regions is a fractal curve. Every tangent point between two circles is a point of the fractal limit set — the set of points never covered by any circle, which has measure zero but Hausdorff dimension ≈ 1.305.

### How This Works — Lissajous Figures

**Core Idea**
A Lissajous figure is the trajectory traced by a point whose $x$ and $y$ coordinates oscillate sinusoidally at (generally different) frequencies. The resulting closed curves — named after Jules Antoine Lissajous, who demonstrated them optically in 1857 — appear in oscilloscope displays, mechanical systems, and signal processing.

**Mathematics**
The parametric equations are:
$$x(t) = \sin(a\,t + \delta), \qquad y(t) = \sin(b\,t)$$
where $a$ and $b$ are the frequency integers and $\delta$ is the phase difference. The curve is **closed** (periodic) whenever $a/b$ is rational — which it always is for integer $a, b$. The full period is:
$$T = \frac{2\pi}{\gcd(a,b)}$$
The shape depends critically on the phase $\delta$: at $\delta = 0$ the curve is a straight line (or an oblique ellipse for $a \neq b$); at $\delta = \pi/2$ a symmetric, "knotted" figure emerges. The number of lobes equals $a$ in one direction and $b$ in the other.

**Logic in the Code**
1. A time vector $t \in [0, 2\pi/\gcd(a,b))$ is sampled uniformly with `n_pts` points — exactly one full period, ensuring the curve closes perfectly.
2. $x(t)$ and $y(t)$ are computed via NumPy broadcasting (no Python loop over samples).
3. Consecutive coordinate pairs are stacked into a `(N-1, 2, 2)` segment array and rendered by `LineCollection`, coloring each segment by its position in $t$ to trace the trajectory's progression.

**Interesting Properties**
When $a/b$ is irrational (approximated with large integers), the curve never closes and densely fills an ellipse over infinite time. Setting $a = b$ and varying $\delta$ traces a circle ($\delta = \pi/2$) → ellipse → line ($\delta = 0$) — the entire family of $\delta$-parameterized curves is used in phase-comparison oscilloscopes. The Lissajous family with $a = b \pm 1$ produce the familiar "figure-8" shapes.

### How This Works — Rose Curves

**Core Idea**
Rose curves (rhodonea curves) are polar curves whose petals emerge from a simple cosine equation. The ratio of two integers determines how many petals appear and how the curve winds around the origin. First studied by Luigi Guido Grandi in 1728, they are among the most elegant examples of how a single parameter change can radically transform a geometric shape.

**Mathematics**
The polar equation is:
$$r = \cos\!\left(\tfrac{p}{q}\,\theta\right)$$
where $p$ and $q$ are coprime integers (reduced to lowest terms). To guarantee closure the parameter must sweep:
$$\theta \in \left[0,\; 2\pi q\right]$$
Petal count depends on the parity of $p$ and $q$: integer $k = p/q$ gives $k$ petals when $k$ is odd and $2k$ when $k$ is even. For non-integer rational $p/q$ the count formula is more complex, but the curves always close after exactly $q$ full revolutions and produce striking layered multi-petal shapes.

**Logic in the Code**
1. $p$ and $q$ are reduced by their GCD so sliders always produce valid coprime ratios.
2. A uniform $\theta$ grid over $[0, 2\pi q]$ is generated; $r = \cos(p/q \cdot \theta)$ is evaluated; Cartesian coordinates follow from $x = r\cos\theta$, $y = r\sin\theta$. Negative $r$ values fold back, tracing petals on the opposite side.
3. A `LineCollection` colors each tiny arc segment by its position in $\theta$, creating a rainbow gradient that reveals the winding order of the petals.

**Interesting Properties**
Rose curves are a subset of the broader family of **harmonograph** curves and share the same parametric structure as Lissajous figures (with $y$ replaced by a radial coordinate). Irrational ratios $p/q$ (approximated by large integers) produce curves with an astronomically large period that appear to fill a disk with dense overlapping petals before eventually closing — a finite analogue of an irrational orbit on a torus.


### How This Works — Chaos Attractor (Lorenz)

**Core Idea**
The Lorenz system is a simplified model of atmospheric convection derived by Edward Lorenz in 1963. Despite having only three variables and three parameters, it exhibits **sensitive dependence on initial conditions** — the butterfly effect — producing a non-repeating trajectory that winds forever around two lobes of a strange attractor without ever crossing itself.

**Mathematics**
The system of ODEs is:
$$\frac{dx}{dt} = \sigma(y - x), \qquad \frac{dy}{dt} = x(\rho - z) - y, \qquad \frac{dz}{dt} = xy - \beta z$$
The classical parameters $\sigma = 10$, $\rho = 28$, $\beta = 8/3$ produce chaotic dynamics. The attractor lives in a bounded region of 3D space; the $x$–$z$ projection — the "butterfly wings" view — is the most recognizable cross-section. Lyapunov exponent $\lambda_1 \approx 0.91$ quantifies the exponential divergence of nearby trajectories.

**Logic in the Code**
1. The trajectory is integrated via classical **4th-order Runge–Kutta (RK4)** using Python floats in a tight loop for speed (no per-step NumPy overhead).
2. All three stages of each RK4 step are inlined as explicit scalar operations; the 3 arrays `traj_x/y/z` are pre-allocated and filled element by element.
3. The first `skip` steps (transient) are discarded; the remaining $x$–$z$ coordinates form a `(N, 2)` path that is split into `LineCollection` segments colored by time, so the early orbit (cooler hues) and later orbit (warmer hues) are visually distinct.
4. The `σ`, `ρ`, `β` sliders let users explore parameter space: reducing `ρ` below ~24.74 destroys chaos and the trajectory converges to a fixed point; raising it re-enters chaos.

**Interesting Properties**
The Lorenz attractor has a fractal cross-section: slicing it with a plane perpendicular to the orbit reveals a Cantor-like set rather than a smooth curve. Its Hausdorff dimension is approximately 2.06 — just barely more than a surface. Lorenz's 1963 discovery launched the modern study of chaos theory and nonlinear dynamics.


### How This Works — Wave Interference Pattern

**Core Idea**
When two or more coherent waves from point sources overlap, they add algebraically — **constructive interference** where crests meet (bright bands) and **destructive interference** where a crest meets a trough (dark bands). The result is a standing fringe pattern that encodes the geometry of the source arrangement. This principle underlies double-slit experiments, radio antenna arrays, and optical holography.

**Mathematics**
Each source $i$ emits a circular wave. At a field point $(x, y)$, the amplitude contributed by source $i$ is:
$$A_i(x,y) = \cos(k\, d_i), \qquad d_i = \sqrt{(x-x_i)^2+(y-y_i)^2}$$
where $k = 2\pi/\lambda$ is the wavenumber. The total field is the **superposition**:
$$A(x,y) = \sum_{i=1}^{n} A_i(x,y)$$
This assumes equal-amplitude, equal-phase, monochromatic sources (fully coherent). The fringe spacing near the centre scales as $\lambda / \theta$ where $\theta$ is the angle subtended by the source pair, matching the classical Young's fringe formula.

**Logic in the Code**
1. $n$ source positions are drawn from a seeded RNG inside a circle of radius 70 units (coordinate system spans ±100), so changing the `Seed` slider gives a new random arrangement.
2. A $\text{res} \times \text{res}$ pixel grid is generated once via `np.meshgrid`; for each source, `np.hypot` computes all distances in a single vectorised call. The `np.maximum(d, 0.5)` guard prevents a numerical singularity at the source location.
3. The summed field is normalised to $[-1, 1]$ and rendered with `imshow` using the selected colormap (diverging palettes like *Arctic Aurora* show constructive maxima and destructive minima most clearly). Source positions are overlaid as white `+` markers.
4. The `Wavelength` slider (in grid units spanning ±100) directly controls $\lambda$; smaller values produce denser fringes with more visible concentric rings per source.

**Interesting Properties**
With two sources separated by distance $d$, the path-length difference to any far-field point is $d\sin\theta$; bright fringes occur when this equals $m\lambda$ (integer multiples). Increasing the number of sources from 2 to $n$ narrows the principal maxima and introduces $n-2$ secondary maxima between them — the same principle used in phased-array radar and Wi-Fi beamforming to concentrate transmitted power in a specific direction.

### How This Works — Hypocycloid & Epicycloid

**Core Idea**
A hypocycloid is the curve traced by a point fixed to a small circle that rolls *inside* a larger fixed circle. An epicycloid traces a point on a circle rolling *outside* the fixed circle. By varying the ratio of radii and the pen distance, an enormous family of closed, star-like, and petal curves emerges — from the simple cardioid (epicycloid, r = R) to the sharp-pointed astroid (hypocycloid, r = R/4).

**Mathematics**
For a fixed circle of radius $R$ and a rolling circle of radius $r$ with the pen at distance $d$ from the rolling circle's center:

*Hypocycloid (hypotrochoid):*
$$x(t) = (R-r)\cos t + d\cos\!\left(\frac{R-r}{r}\,t\right), \qquad y(t) = (R-r)\sin t - d\sin\!\left(\frac{R-r}{r}\,t\right)$$

*Epicycloid (epitrochoid):*
$$x(t) = (R+r)\cos t - d\cos\!\left(\frac{R+r}{r}\,t\right), \qquad y(t) = (R+r)\sin t - d\sin\!\left(\frac{R+r}{r}\,t\right)$$

The curve closes after $T = 2\pi \cdot R / \gcd(R, r)$ radians of $t$ (using integer approximations of $R$ and $r$). When $d = r$ the pen rides on the rim and the generic curve specializes to a true hypo- or epicycloid; for $d \neq r$ it is a *trochoid*.

**Logic in the Code**
1. Integer approximations of $R$ and $r$ compute the period via `gcd` so the curve closes cleanly.
2. `np.linspace` generates $n$ evenly-spaced $t$ values over the full period; both $x$ and $y$ are evaluated in a single NumPy broadcast.
3. Consecutive coordinate pairs form a `LineCollection` with a palette gradient mapped along $t$, so the winding order of the curve is visible as a color progression.
4. The **Type** dropdown switches between hypo- and epi- formulas; all three sliders ($R$, $r$, $d$) are live and immediately change curve topology.

**Interesting Properties**
Special cases: $r = R/4$ → astroid (4-pointed star); $r = R/3$ → deltoid (3-pointed); epi with $r = R$ → cardioid. When $R/r$ is irrational the curve never closes and densely fills an annulus — analogous to an irrational rotation on a torus. The Spirograph toy marketed from 1965 onward is a physical implementation of the hypotrochoid family.

---

### How This Works — Truchet Tiles

**Core Idea**
In 1704, Sébastien Truchet described a simple square tile with a diagonal stripe and observed that randomly orienting such tiles across a grid produces surprisingly rich, organic-looking patterns — including winding rivers, maze corridors, and Celtic knotwork motifs. The arc variant (two quarter-circle arcs per tile) is the modern standard and generates flowing, coral-like networks.

**Mathematics**
Each tile occupies a unit square $[0,1]^2$. In the arc variant two quarter-circles of radius $\tfrac{1}{2}$ connect the midpoints of adjacent sides:
- **Orientation 0:** arc centred at $(0, 0)$ connecting the bottom and left midpoints; arc centred at $(1, 1)$ connecting the top and right midpoints.
- **Orientation 1:** arc centred at $(1, 0)$ and arc centred at $(0, 1)$.

A uniformly random binary matrix (one bit per tile) selects the orientation. For an $n \times n$ grid there are $2^{n^2}$ possible configurations, but all typical realizations share the same visual statistics.

**Logic in the Code**
1. `np.random.default_rng(seed).integers(0, 2, (n, n))` generates the orientation matrix in one call.
2. For each cell, the correct pair of `matplotlib.patches.Arc` objects (or diagonal line segments for the "Diagonals" style) is added to the axes. Color is sampled from the palette gradient indexed by row, giving a horizontal rainbow banding effect.
3. The "Mixed" style uses arcs in even-parity cells ($(row + col) \bmod 2 = 0$) and diagonals in odd-parity cells, producing a hybrid texture.
4. All arcs are drawn with `ax.add_patch` — no rasterization — so the output is fully vector and exports cleanly at any DPI.

**Interesting Properties**
At the percolation threshold (50% probability per orientation), connected arc-paths form a fractal cluster that spans the grid. Smith, Dimond, and Truchet (independently) showed that the expected length of a connected path scales as $O(n^{4/3})$ — the same universality class as 2D percolation. The "Smith chart" used in RF engineering is a distant relative of this tile family.

---

### How This Works — Hexagonal Grid Art

**Core Idea**
The regular hexagonal lattice is the unique tiling of the plane that maximizes the ratio of enclosed area to perimeter — the same reason honeycombs use hexagons. By assigning colors to each cell based on its axial coordinates, a wide variety of interference patterns, gradients, and geometric motifs emerge from the same underlying tessellation.

**Mathematics**
Hexagonal grids are naturally described in *axial coordinates* $(q, s)$ where $r = -q - s$. For a pointy-top layout with cell size $a$, the Cartesian center of cell $(q, s)$ is:
$$x = a\sqrt{3}\!\left(q + \tfrac{s}{2}\right), \qquad y = a \cdot \tfrac{3}{2} \cdot s$$
The six corners of a hexagon at center $(c_x, c_y)$ are:
$$\left(c_x + a\cos\!\left(\frac{\pi k}{3} + \frac{\pi}{6}\right),\; c_y + a\sin\!\left(\frac{\pi k}{3} + \frac{\pi}{6}\right)\right), \quad k = 0,\ldots,5$$
A grid of $r$ rings contains $(3r^2 + 3r + 1)$ hexagons — 127 cells at $r = 6$, 1141 at $r = 18$.

**Logic in the Code**
1. The double loop over $(q, s) \in [-\text{rings}, \text{rings}]^2$ keeps only cells where $|{-q-s}| \leq \text{rings}$, correctly forming a hexagonal outline.
2. Four coloring modes map each cell to a value $t \in [0,1]$: **Distance** (Euclidean distance from origin), **Angle** (azimuth), **Checkerboard** ($(q+s+r) \bmod 3$, producing a 3-color tiling), **Random** (seeded RNG).
3. Each $t$ value indexes the selected colormap; cells are drawn as filled `Polygon` patches with a thin semi-transparent edge to show the grid structure.
4. The `Hex Size` slider scales all coordinates uniformly; combined with `Rings`, it controls the visual density independently of window size.

**Interesting Properties**
The axial coordinate system makes neighbor queries trivial: the six neighbors of $(q, s)$ are exactly $(q \pm 1, s)$, $(q, s \pm 1)$, $(q+1, s-1)$, $(q-1, s+1)$ — no trigonometry required. The checkerboard coloring with 3 colors is the unique proper 3-coloring of the hexagonal lattice, isomorphic to the three-state Potts model studied in statistical mechanics.

---

### How This Works — Spirograph Generator

**Core Idea**
The Spirograph toy (Kenner, 1965) uses physical gears to draw hypotrochoid curves. The pen rides inside a hole in a small plastic gear rolling around the inside of a fixed ring gear. By stacking multiple curves with increasing pen distances, the generator creates the layered, overlapping petal patterns seen in commercial Spirograph art packs.

**Mathematics**
Each layer draws a hypotrochoid with fixed $R$ (outer teeth) and $r$ (inner teeth) but a different pen distance $d_\ell$:
$$d_\ell = r \cdot \left(0.3 + 0.7 \cdot p \cdot \frac{\ell+1}{L}\right), \quad \ell = 0, \ldots, L{-}1$$
where $p$ is the **Pen Ratio** slider and $L$ is the number of layers. This linearly spaces the pen from $0.3r$ (close to the gear center, tight loops) to $p \cdot r$ (near the rim, wide loops).

The curve closes after $T = 2\pi \cdot R / \gcd(R, r)$ radians; integer tooth counts guarantee rational $R/r$ and exact closure.

**Logic in the Code**
1. Teeth counts are converted to floating-point radii; `gcd` gives the exact period.
2. A loop over `layers` computes each hypotrochoid independently using vectorized NumPy; consecutive pairs become `LineCollection` segments.
3. Each layer receives a distinct color from the palette gradient, so inner and outer layers have contrasting hues and the overlap region blends visually.
4. Alpha is set to 0.80 per layer so overlapping regions darken naturally, mimicking the ink accumulation of a real physical spirograph.

**Interesting Properties**
When $\gcd(R, r) = 1$ (coprime teeth), the curve must complete $R$ full rotations of the inner gear before closing — producing the maximum number of petals ($R$). Composite $R/r$ ratios close sooner and produce fewer, wider petals. The limiting case $r \to R$ (gear fills the ring) traces a straight line — the degenerate Tusi couple used in medieval Islamic astronomy to convert circular motion to linear.

---

### How This Works — Parametric Curve Art

**Core Idea**
Parametric curves define position as a pair of functions $(x(t), y(t))$ of a single parameter $t$. Unlike Cartesian equations $y = f(x)$, they naturally represent multi-valued, self-intersecting, and looping paths. This pattern is a curated gallery of eight famous named curves, each with distinct mathematical origins but all renderable by the same `LineCollection` pipeline.

**Mathematics — Selected Curves**

| Curve | Equations | Origin |
|---|---|---|
| **Butterfly** | $x = \sin t \cdot e(t)$, $y = \cos t \cdot e(t)$, $e = e^{\cos t} - 2\cos 4t - \sin^5(t/12)$ | Temple H. Fay, 1989 |
| **Astroid** | $x = \cos^3 t$, $y = \sin^3 t$ | Roemer & Bernoulli, 1674 |
| **Superellipse** | $x = \text{sgn}(\cos t)|\cos t|^{2/n}$, $y = \text{sgn}(\sin t)|\sin t|^{2/n}$, $n=2.5$ | Gabriel Lamé, 1818 |
| **Trisectrix** | $r = 1 + 2\cos\theta$ (polar) | MacLaurin, 1742 |
| **Folium of Descartes** | $x = 3a\tan t/(1+\tan^3 t)$, $y = 3a\tan^2 t/(1+\tan^3 t)$ | Descartes, 1638 |

The Epitrochoid Star ($R=5$, $r=3$, $d=5$) and Hypotrochoid Flower ($R=7$, $r=3$, $d=6$) are trochoid special cases already described under Pattern 16.

**Logic in the Code**
1. Each curve name maps to a `(t_min, t_max)` range chosen so the curve completes exactly one visual cycle without excess overlap.
2. `_compute_curve` evaluates the parametric equations as NumPy arrays — fully vectorized, no Python loop over points.
3. A finite-value mask (`np.isfinite`) removes any NaN/Inf values that arise at algebraic singularities (notably the Folium's asymptote at $\tan t = -1$).
4. The resulting segments are colored by $t$-position along the gradient, revealing the temporal winding order of the curve — where it starts, how it self-intersects, and where it ends.

**Interesting Properties**
The Butterfly curve ($e^{\cos\theta} - 2\cos 4\theta - \sin^5(\theta/12)$) was discovered empirically by Temple Fay while exploring polar-to-parametric transformations; it has no known closed-form area or arc-length. The Superellipse with $n = 2$ is a circle; $n < 2$ gives a "squircle" (between square and circle); the limit $n \to \infty$ approaches a perfect square. Danish designer Piet Hein used superellipses ($n = 2.5$) to design Sergels Torg plaza in Stockholm in 1959.

### How This Works — Cherry Blossom Particle Scene

**Core Idea**
A cherry blossom scene combines two elements: a procedural tree built by recursive binary branching, and a particle system of petals scattered across the canvas. Wind displaces petals horizontally in proportion to their height — petals near the top have fallen farther and drifted more. The result captures the ephemeral, gravity-driven cascade of *hanami* (flower viewing).

**Mathematics**
The recursive branching follows the same geometric principle as a binary fractal tree. Each branch of length $\ell$ at angle $\theta$ spawns two children at angles $\theta \pm \delta$ with length $\ell \cdot r$ where $r \approx 0.67$ is the decay ratio. With depth $d$, the maximum number of segments is $2^d - 1$ (plus occasional third branches for bushy variation).

Petal horizontal displacement due to wind is modeled as:
$$x' = x + w \cdot (1 - y) \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{U}(-0.3, 0.3)$$
where $w$ is the Wind control and $(1 - y)$ ensures petals near the top (large fall distance) drift more than those near the ground.

**Logic in the Code**
1. The `_branch` function accumulates all segments into a list before drawing. A single `LineCollection` draws all branch segments with per-segment colors and linewidths in one call — avoiding thousands of individual `ax.plot` calls.
2. Petal positions are computed via fully vectorized NumPy broadcasting: `n_petals` random $(x, y)$ points, shifted by the wind formula, then clipped to canvas bounds.
3. Petal RGB colours are linearly interpolated from deep rose $(1, 0.35, 0.50)$ to pale blush $(1, 0.85, 0.90)$ using a per-petal hue parameter drawn from $\mathcal{U}(0,1)$.
4. A separate ground scatter plots smaller, semi-transparent petals in the bottom 5.5% of the canvas to simulate accumulation.

**Interesting Properties**
The "third branch" added with 35% probability when depth $> 3$ breaks perfect binary symmetry, producing the irregular silhouette characteristic of real *Prunus serrulata* trees. Increasing wind past 0.7 causes petals to cluster on the leeward side — a qualitatively correct simulation of how wind-driven cherry blossom showers behave in nature.

### How This Works — Procedural Tree Generator

**Core Idea**
A procedural tree is generated by direct recursive geometric branching — no string rewriting, no turtle grammar. At each node a trunk segment gives birth to $n$ child branches, each rotated by a fraction of the configured angle. Small random jitter on each branch's angle and length makes the result organic rather than mechanical. This approach is faster than L-Systems for interactive use and allows more direct control over tree shape.

**Mathematics**
Given a parent branch of length $\ell$ at heading $\theta$ and branching count $n$, the child headings are evenly spaced in the range $[\theta - \phi/2,\;\theta + \phi/2]$:
$$\theta_i = \theta - \phi/2 + \phi \cdot \frac{i}{n-1}, \qquad i = 0, \ldots, n-1$$
where $\phi$ is the Branch Angle parameter. Each child has length:
$$\ell' = \ell \cdot (r + \varepsilon), \qquad \varepsilon \sim \mathcal{U}(-0.03, 0.03)$$
where $r$ is the Length Decay slider. Total segment count is bounded by $n^d$ where $d$ is the depth.

**Logic in the Code**
1. All segments are accumulated in lists (`seg_list`, `t_list`, `lw_list`) without drawing. The recursion only builds data structures.
2. A single `LineCollection` draws all segments simultaneously with per-segment colors (from the selected palette mapped to tree depth $t = 1 - d/\text{depth}$) and per-segment linewidths (thicker near the trunk). This replaces tens of thousands of individual `ax.plot` calls.
3. The palette gradient runs from trunk (near palette start) to tip (near palette end), so the colour encodes tree anatomy: darker base, brighter growing tips.
4. The `_draw` function is bounded to avoid runaway recursion: it exits when segment length drops below 0.005 canvas units, regardless of depth.

**Interesting Properties**
With `n_branches = 2` and `branch_angle ≈ 25°`, the tree resembles a *Populus* poplar. Increasing to `n_branches = 3` and `branch_angle ≈ 40°` produces oak-like spreading crowns. At `length_decay = 0.5` the tree converges rapidly and stays compact; at `0.85` it spreads into a sprawling canopy — illustrating how a single allometric coefficient shapes an entire plant architecture.

### How This Works — Reaction-Diffusion (Turing Patterns)

**Core Idea**
Reaction-diffusion systems describe how chemical species spread through space and react with each other. Alan Turing showed in 1952 that two interacting chemicals — an *activator* and an *inhibitor* — can spontaneously break spatial symmetry and produce the spots, stripes, and labyrinthine patterns seen on animal skins, fish fins, and seashells. The Gray-Scott model is the canonical computational formulation.

**Mathematics**
The Gray-Scott model tracks two concentrations, $A$ (substrate) and $B$ (product):
$$\frac{\partial A}{\partial t} = D_A \nabla^2 A - AB^2 + f(1 - A)$$
$$\frac{\partial B}{\partial t} = D_B \nabla^2 B + AB^2 - (f + k)B$$
The reaction $A + 2B \to 3B$ converts the substrate autocatalytically: one molecule of $B$ recruits two more, turning $A$ into $B$. The feed parameter $f$ replenishes $A$ from a reservoir; the kill parameter $k$ removes $B$. The diffusion constants $D_A = 1.0$ and $D_B = 0.5$ ensure the activator ($B$) diffuses more slowly than the substrate — a necessary condition for pattern formation. Different $(f, k)$ coordinates produce qualitatively different morphologies: spots at $(0.037, 0.060)$, worms at $(0.025, 0.060)$, mazes at $(0.029, 0.057)$.

**Logic in the Code**
1. The grid is initialized with $A = 1$ and $B = 0$ everywhere, then a set of small random square patches are seeded with $A = 0.5$, $B = 0.25$ to break spatial symmetry.
2. The discrete Laplacian is computed at each step via four `np.roll` calls (one per compass direction): $\nabla^2 A \approx A_{i-1,j} + A_{i+1,j} + A_{i,j-1} + A_{i,j+1} - 4A_{i,j}$, with periodic boundary conditions.
3. The reaction and diffusion terms are computed fully vectorized across the $N \times N$ array; `np.clip` keeps both fields in $[0, 1]$ after each step.
4. After all iterations, the $B$ field is rendered via `imshow`; the palette colormap maps concentration to colour.

**Interesting Properties**
The $(f, k)$ parameter space has a rich "phase diagram": crossing the Turing instability boundary flips the pattern from uniform equilibrium to spatial structure. The same Gray-Scott equations that produce animal skin patterns also describe electrode corrosion, chemical oscillators (BZ reaction), and some bacterial colony growth patterns.

### How This Works — Flocking Birds (Boids Lite)

**Core Idea**
Boids (Bird-oid Objects) is Craig Reynolds' 1986 model showing that coherent flocking behavior emerges from three simple local rules applied independently by each agent — no central controller, no global information. The resulting collective motion closely matches the murmurations of starlings, schools of fish, and swarms of insects.

**Mathematics**
Each boid $i$ at position $\mathbf{p}_i$ with velocity $\mathbf{v}_i$ applies three steering forces at each time step:

**Separation** — avoid crowding neighbors within radius $r_s$:
$$\mathbf{f}_s = \sum_{j \in \mathcal{N}_s(i)} (\mathbf{p}_i - \mathbf{p}_j), \quad \text{normalized}$$

**Alignment** — steer toward average heading of neighbors within radius $r_a$:
$$\mathbf{f}_a = \frac{1}{|\mathcal{N}_a|} \sum_{j \in \mathcal{N}_a(i)} \mathbf{v}_j, \quad \text{normalized}$$

**Cohesion** — steer toward center of mass of neighbors within radius $r_c$:
$$\mathbf{f}_c = \frac{1}{|\mathcal{N}_c|} \sum_{j \in \mathcal{N}_c(i)} \mathbf{p}_j - \mathbf{p}_i, \quad \text{normalized}$$

The combined update is $\mathbf{v}_i \mathrel{+}= (w_s \mathbf{f}_s + w_a \mathbf{f}_a + w_c \mathbf{f}_c) \cdot v_{\max}$, then speed is clamped to $v_{\max}$.

**Logic in the Code**
1. Pairwise differences are computed as a single $(N, N, 2)$ tensor: `diff = pos[:, None] - pos[None]`. Minimum-image periodic wrapping (`diff -= np.round(diff)`) makes the space toroidal.
2. Neighbor masks are Boolean $(N, N, 1)$ broadcasts applied to `diff` and `vel[None]` in one NumPy multiply-sum, avoiding all Python loops over boids.
3. Each force vector is normalized to unit length before scaling by its weight and $v_{\max}$, so the weight sliders have consistent effect regardless of local crowd density.
4. The final snapshot is rendered as a `quiver` plot: arrow direction = normalised velocity, arrow color = speed (mapped through the selected palette).

**Interesting Properties**
The three radii satisfy $r_s < r_a < r_c$ — a necessary ordering. If $r_s \geq r_a$ boids cannot align without first repelling each other, causing the flock to disintegrate. Setting $w_s \gg w_a, w_c$ produces an individual-avoidance swarm with no coherent direction; setting $w_c \gg$ others collapses the flock into a single spinning clump.

### How This Works — Lightning Bolt Generator

**Core Idea**
Lightning is a plasma discharge that travels from cloud to ground via the path of least electrical resistance. In reality this path is determined by the probabilistic dielectric breakdown of air — each step is biased toward lower potential but includes random components. The midpoint displacement algorithm captures this stochastic branching structure: a straight path is iteratively perturbed and split into zigzag channels, with random branches splitting off at each level.

**Mathematics**
Given a segment from $\mathbf{p}_1$ to $\mathbf{p}_2$, the midpoint is displaced perpendicular to the segment:
$$\mathbf{m} = \frac{\mathbf{p}_1 + \mathbf{p}_2}{2} + \lambda \cdot \|\mathbf{p}_2 - \mathbf{p}_1\| \cdot \hat{\mathbf{n}} \cdot \varepsilon$$
where $\hat{\mathbf{n}}$ is the unit normal to the segment, $\varepsilon \sim \mathcal{U}(-1, 1)$, and $\lambda$ is the Roughness parameter. Recursing to depth $d$ halves the segment length each time; the total number of base segments is $2^d$. Branches are spawned at each recursion level with probability $p_{\text{branch}}$ — they are shorter-lived (reaching only depth $d - 2$) so they taper convincingly.

**Logic in the Code**
1. The main `_bolt(p1, p2, d, intensity)` function recurses exactly twice per call (left and right half-segments), building a binary tree of segments at the base case ($d = 0$).
2. At each level, with probability `branch_prob` a side channel is launched from the current midpoint toward a randomly deflected endpoint. Its intensity is scaled by 0.45 so branches render thinner and dimmer than the main bolt.
3. All accumulated segments are drawn after recursion completes: a narrow bright core `(0.7+0.3i, 0.8+0.2i, 1.0)` with alpha ∝ intensity, plus a wide low-alpha glow pass in blue to simulate plasma emission.
4. The Roughness slider directly scales $\lambda$: near zero gives a nearly straight bolt; near 0.8 gives chaotic, fractal zigzags typical of megavolt discharges.

**Interesting Properties**
The midpoint displacement algorithm is the same technique used to generate fractal terrain (Diamond-Square) — the same mathematical structure underlies both lightning and mountain ridges. Real lightning exhibits a Hausdorff dimension of approximately 1.7, similar to the dimension of DLA (Diffusion-Limited Aggregation) clusters, because both processes are governed by Laplacian growth.

### How This Works — Snowflake Crystal Growth

**Core Idea**
Real snowflakes grow by water vapour depositing onto an ice crystal. Because hexagonal ice has six-fold symmetry and the local temperature and humidity are nearly identical across the tiny crystal, all six arms grow almost identically — producing the famous six-fold symmetric, yet uniquely detailed, snowflake forms. The simulation captures this by explicitly enforcing six-fold symmetry in a recursive arm-and-branch structure.

**Mathematics**
Each of the six arms grows from the center at angle $k\pi/3$, $k = 0, \ldots, 5$. Along each arm, the main spine recurses with scale factor $r_{\text{arm}}$ (the Arm Scale slider):
$$\ell_{d} = \ell_0 \cdot r_{\text{arm}}^{\,d}$$
At each spine node, two side branches grow at angles $\pm(\pi/3 + \delta_d)$ from the spine direction, where $\delta_d$ is a small seeded jitter providing organic variation without breaking overall six-fold symmetry. Side branches use scale factor $r_{\text{branch}}$ (the Branch Scale slider):
$$\ell_{\text{branch}} = \ell_{\text{spine}} \cdot r_{\text{branch}}$$
Side branches also recurse, producing sub-branches — giving the characteristic hierarchical complexity.

**Logic in the Code**
1. The `_arm(x, y, angle, length, d)` function accumulates all segment data (`(x1,y1), (x2,y2), t`) into lists without drawing. This separates geometry computation from rendering.
2. After all six arms are traced, a single `LineCollection` renders all segments with per-segment colors (palette mapped to depth fraction $t = 1 - d/(depth+1)$) and per-segment linewidths (trunk thick, tips thin).
3. A second `LineCollection` with $5\times$ wider linewidth and $\approx 6\%$ alpha draws a soft glow halo over the crystal, simulating refracted light.
4. The per-depth jitter array `branch_off` is pre-computed from the seeded RNG before recursion begins, guaranteeing that all six arms receive identical jitter (correct six-fold symmetry) while still looking organic.

**Interesting Properties**
Real snowflake diversity arises because each crystal's history of temperature and humidity varies along its growth path — so two crystals growing a millimetre apart can look completely different, even though both are perfectly six-fold symmetric. At the Branch Scale boundary $r_{\text{branch}} = 1.0$, side branches grow as long as the spine — producing a fractal where every sub-branch looks like the whole snowflake (exact self-similarity).

### How This Works — Leaf Venation Simulation

**Core Idea**
Leaf veins form through a process called auxin canalization: the plant hormone auxin flows from the leaf margin inward, and wherever auxin concentration is high, the cell wall stiffens and becomes a conducting channel. The space colonization algorithm by Runions et al. (2007) models this by placing attractor points (representing auxin sources) throughout the leaf and growing vein segments toward whichever attractors are nearest. Once a vein reaches an attractor's kill radius, that source is consumed and vein growth pivots toward the next cluster.

**Mathematics**
Let $\mathbf{A} = \{a_1, \ldots, a_M\}$ be the attractor points and $\mathbf{N} = \{n_1, \ldots, n_K\}$ the current vein node positions. At each iteration:

1. For each attractor $a_i$, find its closest node: $n^*(i) = \arg\min_j \|a_i - n_j\|$.
2. Remove attractors within the kill radius: $\{a_i : \|a_i - n^*(i)\| \leq r_k\}$.
3. For each active node $n_j$ that attracted at least one source, compute the normalized mean direction:
$$\mathbf{d}_j = \frac{\displaystyle\sum_{i : n^*(i) = j} \frac{a_i - n_j}{\|a_i - n_j\|}}{\left\|\displaystyle\sum_{i : n^*(i) = j} \frac{a_i - n_j}{\|a_i - n_j\|}\right\|}$$
4. Grow a new node: $n_{\text{new}} = n_j + s \cdot \mathbf{d}_j$ where $s$ is the step size, if $n_{\text{new}}$ lies inside the leaf ellipse.

**Logic in the Code**
1. Attractors are sampled uniformly inside the ellipse using rejection sampling. The root node starts at the leaf base (bottom of the ellipse).
2. The $(A, N)$ distance matrix is computed each iteration via broadcasting: `diff_an = attractors[:, None] - nodes_arr[None]`. NumPy's `np.argmin` finds the closest node per attractor in one call.
3. New nodes are grown for all active nodes in the same iteration (parallel growth), stored in temporary lists to avoid modifying `nodes` mid-loop.
4. All vein edges are drawn after the simulation via `LineCollection` with colors indexed by tree depth (distance from root) and linewidth proportional to $(1 - t)$ so the mid-rib is thickest.

**Interesting Properties**
The space colonization algorithm naturally produces the hierarchical venation architecture seen in real dicot leaves: a dominant mid-rib, secondary veins branching from it, and fine tertiary reticulation — all without any explicit programming of this hierarchy. Reducing the kill radius $r_k$ produces denser, more reticulate venation (similar to oak leaves); increasing it gives open, parallel venation (similar to grasses).

### How This Works — Fire Particle System

**Core Idea**
A fire particle system models combustion as a cloud of discrete embers, each born at the fuel source, rising through convection, cooling as they age, and fading into smoke. The visual impression of continuous flame emerges from rendering thousands of particles simultaneously at every life stage.

**Mathematics**
Each particle carries an age $a \in [0,1]$, drawn from a Beta$(1, 2.5)$ distribution so young particles (near $a=0$) are far more numerous than dying ones — matching the dense base of a real flame. Position is:
$$x = \mathcal{N}\!\left(0.5,\;\sigma_x(a)\right), \qquad y = a\,(h_{\max} + \delta), \quad h_{\max} = 0.70 + \text{heat}\times 0.22$$
where $\sigma_x(a) = 0.05 + \text{turbulence}\times 0.12\,a$ widens the flame horizontally with age. Colour follows a temperature-to-RGB mapping: green channel $g = \min(1, 1.9(1-a))$ and blue channel $b = \max(0,(1-a-0.65)\times 3)$, producing the white–yellow–orange–red gradient of a blackbody radiator.

**Logic in the Code**
1. `rng.beta(1.0, 2.5, n)` produces age values skewed toward 0, so most particles are near the base (hot and bright).
2. Horizontal spread is proportional to age — older particles have had more time to drift laterally via turbulence.
3. RGBA colour arrays are assembled with pure numpy broadcasting; no per-particle loop.
4. A separate ember cluster (tight Normal spread, age≈0) is overlaid at $y\approx 0$ to represent the hot fuel surface.
5. `ax.scatter` renders all particles in a single draw call with size inversely proportional to age.

**Interesting Properties**
The Beta distribution is the key to realism: a uniform age distribution would make the flame look hollow (equal density everywhere), while Beta$(1, 2.5)$ naturally concentrates particles at the base, matching the luminous cone shape visible in candle flames. Increasing turbulence shifts $\sigma_x$, transitioning from a calm pillar to a storm-tossed torch.

### How This Works — Galaxy Spiral Arms

**Core Idea**
Spiral galaxies exhibit arms that follow logarithmic spirals, a form that appears throughout nature wherever growth occurs at a constant angle relative to the centre. This pattern simulates a barred-spiral galaxy by placing star-like points along multiple logarithmic arms plus a compact central bulge.

**Mathematics**
A logarithmic spiral in polar form is:
$$r(\theta) = r_0\,e^{b\,\theta}$$
where $r_0 = 0.01$ is the inner radius and $b$ is derived so $r$ reaches $r_{\max}=0.46$ over $\theta_{\max} = \text{winding}\times 3\pi$:
$$b = \frac{\ln(r_{\max}/r_0)}{\theta_{\max}}$$
Each arm is rotated by $2\pi k/N_{\text{arms}}$ for arm index $k$. Stars are scattered with Gaussian spread $\sigma(t) = 0.004 + 0.020\,t$ perpendicular to the arm tangent $(-\sin\theta, \cos\theta)$, widening toward the outer edge.

**Logic in the Code**
1. For each arm, $t\in[0,1]$ is sampled uniformly; $\theta = t\,\theta_{\max}$ and $r = r_0 e^{b\theta}$.
2. Cartesian coordinates are computed then perturbed perpendicular to the local tangent direction.
3. A central bulge is added using a half-Normal radial distribution, producing a bright compact nucleus.
4. Star colour transitions from warm yellow-white ($t\approx 0$, inner) to cool blue-white ($t\approx 1$, outer), mimicking stellar population gradients.
5. Point sizes decrease with $t$ so inner stars appear brighter.

**Interesting Properties**
The winding number controls how many turns each arm makes. At high winding (many turns), arms overlap and the galaxy looks like a tight grand-design spiral (think M51); at low winding, it resembles a flocculent open structure. The logarithmic form means the pitch angle $\psi = \arctan(1/b)$ is constant along the arm — a property observed in real galaxies.

### How This Works — Aurora Borealis

**Core Idea**
The aurora borealis forms when charged solar-wind particles excite atmospheric gases along magnetic field lines, producing glowing curtains of light. This pattern simulates multiple translucent curtains using sinusoidally-shaped intensity masks rendered directly onto a raster image.

**Mathematics**
Each curtain's upper edge traces a compound wave:
$$y_{\text{edge}}(x) = y_0 + A\sin\!\left(\frac{2\pi f x}{W} + \phi\right) + 0.4A\sin\!\left(\frac{2\pi\cdot 2.3f x}{W} + 1.7\phi\right)$$
The pixel intensity at row $r$, column $c$ is:
$$I(r,c) = \exp\!\left(-\frac{\max(0,\,r - y_{\text{edge}}(c))}{L}\right) \cdot S(c)$$
where $L$ is the curtain length and $S(c)$ is a shimmer modulation factor. Curtains are composited additively onto the RGB canvas.

**Logic in the Code**
1. A difference matrix `dy[row, col] = row - wave[col]` is computed in one numpy broadcast, covering the full $H\times W$ grid.
2. The exponential decay is applied element-wise — no Python loop over rows.
3. A shimmer sinusoid multiplies each column, producing the flickering vertical striations of a real aurora.
4. Multiple curtains are composited additively, clipped, then gamma-corrected with a 0.55 power curve to lift dark regions.
5. A sparse starfield is painted last at full brightness, visible through the curtain gaps.

**Interesting Properties**
The additive compositing model means overlapping curtains of different colours mix to produce intermediate hues — green+purple yields cyan-tinted columns. Real auroras exhibit exactly this blending where oxygen (green) and nitrogen (blue/red) emission bands overlap.

### How This Works — Underwater Caustics

**Core Idea**
Caustics are the bright, branching light patterns visible on the floor of a swimming pool or shallow sea. They arise because a wavy water surface acts as a lens array, focusing refracted light into moving regions of high intensity. This pattern simulates the interference field that approximates the caustic distribution.

**Mathematics**
The intensity field is modelled as a sum of plane-wave cosines from $N$ random directions:
$$F(x,y) = \sum_{i=1}^{N} \cos(\mathbf{k}_i \cdot \mathbf{r} + \phi_i), \quad \mathbf{k}_i = 2\pi f\,(\cos\alpha_i,\, \sin\alpha_i)$$
where $\alpha_i \sim \mathcal{U}[0, 2\pi)$ and $\phi_i \sim \mathcal{U}[0, 2\pi)$. After normalising $F$ to $[0,1]$, a power law sharpens the peaks:
$$C(x,y) = F(x,y)^{\gamma}, \quad \gamma = \text{sharpness}$$
High $\gamma$ pushes the field toward binary (dark background, thin bright caustic threads).

**Logic in the Code**
1. The full $N\times N$ grid of $(x,y)$ coordinates is computed once via `np.meshgrid`.
2. Each wave's contribution is a single vectorised numpy expression — no spatial loops.
3. After summation the field is normalised, then `np.power` applies $\gamma$ uniformly.
4. The scalar field is mapped to a blue-teal RGB image by independently scaling each channel, giving the characteristic underwater blue cast with bright cyan highlights.

**Interesting Properties**
The interference pattern is deterministic given the wave directions but appears highly irregular — it is a quasi-random field whose statistics resemble real caustics (exponential intensity distribution). Increasing frequency raises the spatial density of caustic threads, while increasing sharpness $\gamma$ transitions the image from smooth gradients to an almost binary light-map.

### How This Works — Sand Dune Erosion

**Core Idea**
Desert sand dunes form and migrate through saltation: wind lifts grains from one location and deposits them downwind. Where the supply exceeds the angle of repose, avalanching redistributes excess sand to restore a stable slope. This pattern simulates both effects on a 2-D heightfield.

**Mathematics**
At each time step the saltation flux from cell $(i,j)$ to its downwind neighbour $(i,j+1)$ is:
$$q_{i,j} = w\,h_{i,j}$$
where $w$ is wind speed. Heights update as:
$$h \leftarrow h - q + \mathrm{roll}(q,\,+1,\,\mathrm{axis}=1)$$
The avalanche rule checks the right-neighbour slope $s_{i,j} = h_{i,j} - h_{i,j+1}$. If $s > \theta_r$ (angle of repose), half the excess transfers:
$$h_{i,j} \leftarrow h_{i,j} - \tfrac{1}{2}(s - \theta_r), \qquad h_{i,j+1} \leftarrow h_{i,j+1} + \tfrac{1}{2}(s - \theta_r)$$
Both operations use `np.roll` for periodic boundary conditions.

**Logic in the Code**
1. The heightfield is initialised with random noise plus a few pre-formed ridges to seed dune formation.
2. Each iteration applies the saltation step (one roll) followed by avalanche passes in both x and y directions.
3. Periodic boundaries (implicit in `np.roll`) let sand wrap around the canvas, simulating an infinite dune field.
4. The normalised heightfield is mapped to a sandy RGB palette: dark ochre in troughs, bright cream at crests — matching the shadowed face and sun-lit slip face of real barchans.

**Interesting Properties**
The interplay between saltation (directional transport) and avalanching (isotropic smoothing) spontaneously organises the random initial field into asymmetric dunes with a gentle windward slope and a steep leeward slip face — the characteristic barchan morphology. Increasing wind speed accelerates migration; reducing the avalanche threshold produces sharper, more angular crests.

### How This Works — Coral Reef Growth

**Core Idea**
Coral polyps grow by budding new individuals at the tips of existing branches, producing the characteristic fan, staghorn, and brain morphologies. This pattern captures that branching growth process using a depth-limited recursive tree whose parameters mimic specific coral species.

**Mathematics**
Each branch of length $\ell$ at depth $d$ spawns $n_{\text{sub}} \in \{2, 3\}$ children at angles evenly distributed over an angular fan $[-\phi, +\phi]$ around the parent direction $\alpha$:
$$\alpha_k = \alpha - \phi + \frac{2k\phi}{n_{\text{sub}}-1} + \varepsilon, \quad \varepsilon \sim \mathcal{U}[-0.08, 0.08]$$
Child length is $\ell_{\text{child}} = \ell \cdot U[0.57, 0.73]$. Recursion terminates at $d=0$ or $\ell < 0.004$. The fan-spread control $\phi \in [0.25,\,0.85]\cdot\pi/2$ governs whether growth is a tight pillar (small $\phi$) or a broad fan (large $\phi$).

**Logic in the Code**
1. An outer loop places $N_{\text{colonies}}$ colonies at random positions along the seafloor.
2. Each colony uses its own colour palette (6 species presets), passed via a default-argument capture to avoid closure aliasing.
3. Segments accumulate in shared lists across all colonies, then are drawn in one `LineCollection` call — far faster than individual `ax.plot` calls.
4. Line width is proportional to depth $d$, so trunks are thick and tips are hairline — matching the tapered morphology of real coral.

**Interesting Properties**
The fan-spread parameter shifts the topology from anisotropic (tree-like, upward) to nearly isotropic (brain coral-like, radial). At extreme fan spread approaching $\pi$, growth folds back on itself, producing the rounded lobes of brain corals like *Diploria* species.

### How This Works — Mushroom Spore Map

**Core Idea**
Mushroom mycelium spreads outward from spore germination points, and cross-sections of the colony boundary map closely to Voronoi cells. This pattern uses a Voronoi distance field to tile the canvas into organic cells, then adds concentric rings and layered-sine noise to create the fibrous, earthy texture of a spore microscopy image.

**Mathematics**
Given $N$ spore centres $\{\mathbf{p}_i\}$, the two nearest-neighbour distances at pixel $\mathbf{q}$ are $d_1(\mathbf{q})$ and $d_2(\mathbf{q})$ (via `scipy.spatial.cKDTree`). Three texture layers are combined:
$$\text{rings}(\mathbf{q}) = \tfrac{1}{2}\bigl(1 + \sin(60\pi f_r d_1)\bigr)$$
$$\text{edge}(\mathbf{q}) = \left(\frac{d_2 - d_1}{d_1 + 0.02}\right)^{0.35}$$
$$\text{field} = \bigl[\text{rings}\cdot(1-\beta) + \text{noise}\cdot\beta\bigr]\cdot\text{edge} + 0.18\,(i_{\text{cell}} \bmod 9)/9$$
where $\beta$ is the noise-blend slider and noise is a three-octave layered-sine approximation to Perlin noise.

**Logic in the Code**
1. The entire pixel grid is queried against the KD-tree in one batched call, returning the two nearest distances and indices.
2. The ring field, edge mask, and noise field are all computed as full $N\times N$ numpy arrays without any Python spatial loop.
3. The cell index $i_{\text{cell}} \bmod 9$ shifts each cell's brightness by a small constant, making individual spore territories visually distinct.
4. The final scalar field is mapped to a three-channel image with different linear ramps per channel, yielding the cream-to-ochre-to-brown mushroom palette.

**Interesting Properties**
When ring frequency is high and noise blend is zero, the pattern resembles tree growth rings. When noise blend is 1.0 the rings vanish and the image looks like a fungal mycelium network stained under a microscope. The two limits represent different biological interpretations of the same Voronoi topology.

### How This Works — Terrain Height Map

**Core Idea**
Terrain generation uses **fractal Brownian motion (fBm)** — a weighted sum of layered noise octaves at geometrically increasing frequencies and decreasing amplitudes. Each octave adds finer detail, producing the characteristic self-similarity of real terrain: mountain ranges resemble foothills, which resemble individual ridges. The finished heightfield is coloured using a **hypsometric** palette that maps elevation bands to geographically meaningful hues.

**Mathematics**
The heightfield is:
$$h(x,y) = \frac{1}{Z}\sum_{k=0}^{n-1} H^k \cdot \sin\!\bigl(2^k f_0\, x + \phi_{k,1}\bigr) \cdot \cos\!\bigl(2^k f_0\, y + \phi_{k,2}\bigr)$$
where $H \in (0,1)$ is the **roughness** (persistence) parameter, $f_0$ is the base frequency, $\phi_{k,1}$ and $\phi_{k,2}$ are per-octave random phases, and $Z = \sum_k H^k$ normalises the sum to $[0,1]$. When $H$ is small, higher octaves are heavily suppressed — producing smooth, low-frequency terrain. As $H \to 1$, all octaves contribute equally — producing rough, fractal surfaces. The Hurst exponent of this process is related to $H$ by $D = 3 - H$ (fractal dimension of the surface).

**Logic in the Code**
1. Each octave draws new random phase offsets from the seeded RNG, ensuring reproducible but visually distinct terrain for every seed value.
2. After the loop, the raw sum is normalised to $[0,1]$ and the elevation is mapped through four Boolean masks: water ($h \leq w$), lowland, midland, and highland — each with a distinct linear colour ramp per RGB channel.
3. `matplotlib.contour` draws the water shoreline at $h = w$ in blue and subtle elevation bands at five equally-spaced heights in translucent black.

**Interesting Properties**
The coastline shape produced by fBm is statistically similar to real coastlines, whose fractal dimension $D \approx 1.2$–$1.3$ was famously analysed by Mandelbrot. Decreasing the water-level slider submerges land until only mountain peaks protrude — the same mechanism that produces archipelagos and drowned river valleys on geologic timescales.

### How This Works — Waterfall Flow

**Core Idea**
A waterfall is water flowing off a cliff edge and falling under gravity to a pool below. The visual signature — a white veil of braided streams — arises from the competition between gravity (pulling straight down) and the Kelvin-Helmholtz instability of the water sheet (causing lateral oscillations). This pattern simulates the effect using individual stream paths driven by gravity with sinusoidal sway, overlaid on a rocky cliff background raster image.

**Mathematics**
Each stream is a discrete-time path. At step $i$, the particle at $(x_i, y_i)$ updates as:
$$x_{i+1} = x_i + A\sin(\omega i + \phi), \qquad y_{i+1} = y_i - s\bigl(1 + k(1 - y_i)\bigr)$$
where $A$ is the sway amplitude, $\phi$ is a per-stream random phase, $s$ is the step size, and the factor $(1 + k(1-y_i))$ provides gravitational acceleration — the stream speeds up as it falls. The sinusoidal term models lateral oscillation of the falling water sheet.

**Logic in the Code**
1. A rocky cliff background is built as an $N \times N$ raster image using five octaves of layered sine noise; the bottom 18% is recoloured dark blue to represent the pool.
2. Each of $n_{\text{streams}}$ particle paths is traced step by step, collecting segment endpoints into shared lists — never drawing mid-loop.
3. A single `LineCollection` renders all segments with per-segment RGBA colour (bright white-blue at the crest, slightly grey at the pool) and per-segment linewidth (thicker at the crest, hairline at the base).
4. A spray cloud uses `ax.scatter` with Beta$(1.5, 3.5)$-distributed $y$-coordinates, concentrating most spray near the pool surface with a few droplets drifting upward.

**Interesting Properties**
The acceleration factor $1 + k(1-y)$ means streams enter the pool faster than they leave the cliff edge, mimicking free-fall under gravity. Real waterfalls exhibit a **hydraulic jump** at the base: a rapid transition from fast, shallow flow to slow, deep flow. This is the physical origin of the turbulent spray cloud, and its radius scales with the incoming flow velocity squared.

### How This Works — Tornado Vortex

**Core Idea**
A tornado is a rapidly rotating column of air connecting a cumulonimbus cloud to the ground. The visible funnel is a condensation cloud that reveals the pressure drop inside the vortex core. This pattern simulates the funnel geometry by distributing particles in a tapered cylinder whose radius is narrow at the top and wide at the base, then twisting their angular positions proportionally to height.

**Mathematics**
In a Rankine vortex the tangential velocity is:
$$v_\theta(r) = \begin{cases} \Omega r & r \leq R_c \quad\text{(solid-rotation core)} \\ \Gamma / (2\pi r) & r > R_c \quad\text{(irrotational outer region)} \end{cases}$$
The funnel radius at normalised height $y \in [0,1]$ is:
$$R(y) = R_{\text{top}} + (1-y)(R_{\text{base}} - R_{\text{top}})$$
A particle at height $y$ receives a twist $\Delta\theta = \omega \cdot y \cdot 2\pi$, where $\omega$ is the Vortex Strength slider. Its 2D projected $x$-coordinate is $x = 0.5 + r\cos(\theta_0 + \Delta\theta)$.

**Logic in the Code**
1. Heights $y$ are drawn uniformly; the local funnel radius follows from the linear taper formula. Each particle's radial distance is a half-Normal centred on the funnel wall, placing debris at the condensation boundary.
2. The twist $\omega \cdot y \cdot 2\pi$ is added to the random azimuth before projection — higher $\omega$ wraps the spiral more tightly.
3. Colour encodes radius (bright white core to dark debris wall) modulated by height (darker near ground, lighter at cloud base), computed entirely with numpy broadcasting.
4. The funnel outline is two polylines $x = 0.5 \pm R(y)$, drawn faintly as a geometric guide.

**Interesting Properties**
Conservation of angular momentum explains why the funnel narrows upward: $r \cdot v_\theta = \Gamma/(2\pi) = \text{const}$, so $v_\theta$ must increase as $r$ decreases — the same mechanism that makes an ice skater spin faster when pulling in their arms. Setting the Funnel Top slider near zero approaches an idealised point vortex, while large values produce a nearly cylindrical pillar of cloud.

### How This Works — Cloud Formation

**Core Idea**
Cumulus clouds form when moist air rises, expands adiabatically, and cools to its dew point at which water vapour condenses into liquid droplets. The characteristic flat cloud base marks the condensation level; the rounded top is limited by convective instability. This pattern simulates the cloud water-content density field using multi-octave persistence noise and composites it over a physically motivated sky gradient.

**Mathematics**
The cloud density field is fractal Brownian motion:
$$N(x,y) = \frac{1}{Z}\sum_{k=0}^{n-1} H^k \cdot \sin\!\bigl(2^k f_0 x + \phi_{k,1}\bigr)\cdot\cos\!\bigl(2^k f_0 y + \phi_{k,2}\bigr)$$
Cloud opacity is derived by thresholding with a power-law lift:
$$\alpha(x,y) = \left(\frac{\max(0,\, N - \tau)}{1 - \tau}\right)^{1/2}, \qquad \tau = 1 - \text{density}$$
Larger density lowers $\tau$, exposing more of the noise field as cloud. The $\tfrac{1}{2}$ exponent softens the hard threshold edge, producing smooth billowing boundaries rather than binary cutoffs.

**Logic in the Code**
1. Each octave is a sine-cosine product with independent random phases; frequency doubles and amplitude scales by `persistence` each step.
2. The sky gradient is a 2D linear ramp — warm hazy horizon to deep blue zenith — computed as numpy arrays over the grid.
3. A shadow layer rolls the cloud alpha 10 rows and 7 columns to simulate sunlight from upper-left, darkening the sky beneath cloud edges by up to 22%.
4. Final compositing: $\text{pixel} = \text{sky} \cdot \text{shadow} \cdot (1-\alpha) + \text{cloud} \cdot \alpha$.

**Interesting Properties**
High persistence (near 0.85) produces large connected cloud masses resembling overcast stratus; low persistence (near 0.35) creates isolated wispy patches resembling cirrus. This maps to the physical correlation length of atmospheric turbulence: high persistence means large-scale convective cells dominate, while low persistence means small-scale granularity breaks up any continuous sheet.

### How This Works — River Delta Branching

**Core Idea**
River deltas form where a river meets standing water, decelerates, and deposits its sediment load. Flow splits into distributary channels wherever a channel becomes too shallow to carry its full discharge — a process called **avulsion**. This pattern models the branching network as a recursive binary tree growing downward from a single trunk, with a fan-spread parameter controlling delta morphology from bird's-foot to arcuate.

**Mathematics**
Each channel of length $\ell$ at recursion depth $d$ bifurcates into two children at angles $\pm\phi/2$ from the parent heading $\alpha$:
$$\alpha_{\text{L}} = \alpha - \phi/2 + \varepsilon, \quad \alpha_{\text{R}} = \alpha + \phi/2 + \varepsilon, \quad \varepsilon \sim \mathcal{U}(-0.06, 0.06)$$
Child length is $\ell' = \ell \cdot r + \eta$ where $r$ is the Length Decay slider. The downward convention is $\Delta x = \ell\sin\alpha$, $\Delta y = -\ell\cos\alpha$, so $\alpha = 0$ is straight down. With depth $d$, the maximum terminal channels is $2^d$.

**Logic in the Code**
1. The recursive `_channel` function accumulates segment data (endpoints, depth fraction $t$, linewidth) into shared lists — no drawing during recursion, keeping geometry separate from rendering.
2. After recursion, a single `LineCollection` draws all channels with per-segment colour: muddy-brown at $t=0$ (trunk) to teal-blue at $t=1$ (sea-reaching distributaries).
3. A sediment fan scatter uses Beta$(1.2, 2.8)$-distributed $y$-coordinates to concentrate deposits near the shoreline.
4. A 28% probability tertiary branch introduces realistic mid-channel avulsions, breaking perfect binary symmetry for a more natural appearance.

**Interesting Properties**
The Fan Spread parameter governs delta morphology: small values produce bird's-foot deltas (like the Mississippi — high sediment supply, low wave energy); large values produce arcuate fan deltas (like the Nile — wave energy redistributes sediment laterally). At the limit $\phi \to \pi$, the tree folds back on itself and loses all net downstream transport.

### How This Works — Moth Wing Pattern

**Core Idea**
Moth wing patterns serve dual ecological functions: **cryptic coloration** (earthy brown-ochre palettes mimic tree bark and dead leaves) and **deimatic display** (prominent eyespots startle predators or direct their strike away from the body). The coloration is produced during metamorphosis by spatially regulated pigment gene expression, generating both concentric band structures and sharply defined eyespot rings.

**Mathematics**
The wing shape is the union of two ellipses. The concentric band field uses the minimum elliptical distance $D = \min(d_{\text{fore}},\, 1.05 \cdot d_{\text{hind}})$:
$$B(x,y) = \tfrac{1}{2}\bigl(1 + \sin(D \cdot k \cdot \pi \cdot s)\bigr)$$
where $k$ is band count and $s$ is pattern scale. Each bilateral eyespot uses Gaussian radial ring functions:
$$E_j(r) = \exp\!\left(-\frac{(r - r_j)^2}{2\sigma_j^2}\right), \quad r = \|(x,y) - (e_x,e_y)\|$$
with four concentric rings: outer light ring, dark ring, inner iris, and dark pupil.

**Logic in the Code**
1. The base field is `bands * 0.65 + noise * 0.35`, mapped to a tan-to-brown RGB palette via three independent linear ramps per channel.
2. Wing veins subtract a narrow Gaussian strip along each radial direction from the body centre: `exp(-perp_dist^2 * 1800) * forward`, multiplied by the wing mask to prevent bleed-through.
3. Eyespots add and subtract four Gaussian ring contributions per spot using `np.where` with the combined wing and radius mask.
4. A bilateral eyespot at $(+0.45, e_y)$ and $(-0.45, e_y)$ for each row in `eye_ys` enforces left-right symmetry — exactly the developmental mechanism used in real Lepidoptera.

**Interesting Properties**
Eyespots function as **deflection marks**: experimental studies with predatory birds show they aim at the prominent false eyes rather than the moth's body, causing survivable wing damage instead of lethal strikes. The concentric ring structure arises from a reaction-diffusion morphogen gradient during wing disc development — the same Turing-pattern mechanism as Pattern 23, operating at millimetre scale within each scale cell.

---

## Section 5 — Export Utilities

Export functions are available via the **💾 EXPORT** button above.
- `export_png(figure, name)` — saves as PNG
- `export_gif(frames, name, fps)` — saves as animated GIF
- `export_mp4(frames, name, fps)` — saves as MP4 (falls back to GIF if ffmpeg unavailable)

All outputs are saved to `visual_engine/exports/`.

### How This Works — Generative Mondrian

**Core Idea**
Piet Mondrian's iconic grid paintings reduce composition to its essentials: thick black lines dividing a white canvas into rectangles, with a few cells flooded in primary red, blue, or yellow. Generative Mondrian recreates this by treating the canvas as a recursive binary space partition tree.

**Mathematics**
At each step every rectangle $(x, y, w, h)$ is split with probability $p = 0.75$. The cut is placed at a random fraction $f \sim \text{Uniform}(0.25,\, 0.75)$ along the longer axis:

$$x_{\text{cut}} = x + f \cdot w \quad \text{(horizontal split if } w \ge h\text{)}$$

After $s$ split rounds the expected number of rectangles grows as $O\bigl((1.5)^s\bigr)$ — exponential growth pruned by the 25 % no-split probability.

**Logic in the Code**
1. A list of rectangles starts with a single unit square.
2. Each iteration replaces every rectangle stochastically: if split, two child rectangles are appended; otherwise the parent is kept unchanged.
3. After all splits, each rectangle is drawn as a `matplotlib.patches.Rectangle` with black edge and `line_width` controlling stroke weight.
4. Colour is sampled from the Mondrian palette (5 whites, 1 red, 1 blue, 1 yellow), so the 5:3 weighting keeps most cells white, matching Mondrian's own compositional ratios.

**Interesting Properties**
Because each split is independent, the composition is statistically self-similar at every scale — zooming into any corner tends to show the same mix of large white and small coloured rectangles. Changing `seed` produces an entirely different layout while preserving this statistical character.


### How This Works — Perlin Noise Painting

**Core Idea**
Fractal Brownian motion (fBm) noise produces smooth, naturalistic textures — clouds, terrain, marble, wood grain — by layering octaves of coherent noise at increasing frequencies and decreasing amplitudes. Unlike white noise, nearby pixels are correlated, giving the characteristic organic look.

**Mathematics**
The fBm field sums $n$ octaves:

$$F(x, y) = \sum_{k=0}^{n-1} \frac{1}{2^k} \cdot N_k\!\left(2^k x,\; 2^k y\right)$$

where $N_k$ is smooth (value) noise at octave $k$. Persistence $= 0.5$ halves amplitude each octave; lacunarity $= 2$ doubles spatial frequency. The series converges because $\sum_{k=0}^{\infty} (1/2)^k = 2$.

**Logic in the Code**
1. For octave $k$, a small random grid of size $(\texttt{scale} \cdot 2^k + 2)^2$ is drawn from `rng.random()`.
2. `scipy.ndimage.zoom` upsamples this grid to full resolution using cubic interpolation (`order=3`), creating spatially correlated smooth values.
3. Each octave is accumulated with weight $1/2^k$; the final result is normalised to $[0, 1]$.
4. The scalar field is rendered via a `LinearSegmentedColormap` built from the chosen palette, mapping noise value to colour.

**Interesting Properties**
The `scale` control changes the base spatial frequency — high values zoom in to show large smooth blobs; low values produce fine-grained detail. Beyond 7–8 octaves, visual improvement is negligible because the highest-frequency layers contribute less than 1 % of total signal energy.
